## 中断

中断的触发是通过抛出 GraphInterrupt 异常实现的，如果用 `try/catch` 捕获，则底层运行时无法感知断电，导致不会中断

#### Human In The Loop

In [ ]:
from typing import TypedDict

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command


class State(TypedDict):
    username: str

def a(state: State):
    name = interrupt("请输入您的姓名")
    return {
        "username": name
    }


builder = StateGraph(State)
builder.add_node(a)
builder.add_edge(START, "a")
builder.add_edge("a", END)
graph = builder.compile(checkpointer=InMemorySaver())

config = {"configurable": {"thread_id": "5-1-1"}}
result = graph.invoke({}, config=config)
print(result)

prompt = result['__interrupt__'][0].value
username = input(prompt)
result = graph.invoke(Command(resume=username), config=config)
print(result)

#### 并行多个中断

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command, interrupt
from typing import TypedDict
from time import sleep


class State(TypedDict):
    username: str
    age: int

def a(state: State):
    name = interrupt("请输入您的姓名")
    return {
        "username": name
    }

def b(state: State):
    sleep(1)
    age = interrupt("请输入您的年龄")
    return {
        "age": int(age)
    }


builder = StateGraph(State)
builder.add_node(a)
builder.add_node(b)
builder.add_edge(START, "a")
builder.add_edge(START, "b")
builder.add_edge("a", END)
builder.add_edge("b", END)
graph = builder.compile(checkpointer=InMemorySaver())

config = {"configurable": {"thread_id": "5-1-2"}}
result = graph.invoke({}, config=config)
print(result)

resume_map = {}
for prompt in result['__interrupt__']:
    resume_map[prompt.id] = input(prompt.value)
result = graph.invoke(Command(resume=resume_map), config=config)
print(result)

#### 审批举例

In [ ]:
from dotenv import load_dotenv
from langchain_deepseek import ChatDeepSeek

load_dotenv(override=True)

model = ChatDeepSeek(
    model="deepseek-flash",
    extra_body={"thinking": {"type": "disabled"}}
)

In [ ]:
from pydantic import BaseModel
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage
from langgraph.types import interrupt, Command
from typing import TypedDict, Any, Literal


class Decision(BaseModel):
    approved: bool
    note: str | None = None

class State(TypedDict):
    topic: str
    poem: str
    answer: Any

def approve_node(state: State) -> Command[Literal["llm_node", "__end__"]]:
    answer = interrupt("是否同意调用模型？", response_schema=Decision)
    return Command(
        goto="llm_node" if answer.approved else "__end__",
        update={"answer": answer}
    )

def llm_node(state: State):
    poem_msg = HumanMessage(f"请写一首关于{state["topic"]}的五言绝句，不用解释内容")
    ai_msg = model.invoke([poem_msg])
    return {
        "poem": ai_msg.content,
    }


builder = StateGraph(State)
builder.add_node(approve_node)
builder.add_node(llm_node)
builder.add_edge(START, "approve_node")
builder.add_edge("llm_node", END)
graph = builder.compile(checkpointer=InMemorySaver())

config = {"configurable": {"thread_id": "5-1-3"}}
result = graph.invoke({"topic": "猫"}, config=config)
print(result)

prompt = result['__interrupt__'][0].value
while True:
    raw_approved = input(prompt)
    if raw_approved in {"是", "否"}:
        break
    print("请输入“是”或“否”")
note = input("备注（可留空）：").strip() or None
resume_value = {
    "approved": raw_approved == "是",
    "note": note,
}

result = graph.invoke(Command(resume=resume_value), config=config)
print(result)